# NATURAL LANGUAGE INFERENCE AND THE DATASET

When there is a need to decide whether one sentence can be inferred from another, or eliminate redundancy by identifying sentences that are semantically equivalent, knowing how to classify one text sequence is insufficient. Instead, we need to be able to reason over pairs of text sequences.

## The Natural Language Inference Dataset

Stanford Natural Language Inference (SNLI) Corpus is a collection of over 500000 labeled English sentence pairs. We will download and store the extracted SNLI dataset.

In [1]:
import os
import torch
from torch import nn
import re
from utils import spy

c:\Users\DELL\.conda\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = spy.download_extract('https://nlp.stanford.edu/projects/snli/snli_1.0.zip', 
                                '../../data', sha1_hash='9fcde07509c7e87ec61c640c1b2753d9041758e4')

### 1. Reading the dataset

The original SNLI dataset contains much richer information than what we really need in our experiments. Thus, we define a function `read_snli` to only extract part of the dataset, then return lists of premises, hypotheses, and their labels.

In [6]:
def read_snli(data_dir, is_train):
    """Read the SNLI dataset into premises, hypotheses, and labels"""
    def extract_text(s):
        s = re.sub('\\(', '', s)
        s = re.sub('\\)', '', s)
        s = re.sub('\\s{2,}', ' ', s)
        return s.strip()
    label_set = {'entailment': 0, 'contradiction': 1, 'neutral': 2}
    file_name = os.path.join(
        data_dir, 'snli_1.0_train.txt' if is_train else 'snli_1.0_test.txt')
    with open(file_name, 'r') as f:
        rows = [row.split('\t') for row in f.readlines()[1:]]
    premises = [extract_text(row[1]) for row in rows if row[0] in label_set]
    hypotheses = [extract_text(row[2]) for row in rows if row[0] in label_set]
    labels = [label_set[row[0]] for row in rows if row[0] in label_set]
    return premises, hypotheses, labels

Now let’s print the first 3 pairs of premise and hypothesis, as well as their labels (“0”, “1”, and “2” correspond to “entailment”, “contradiction”, and “neutral”, respectively ).



In [7]:
train_data = read_snli(data_dir, is_train=True)
for x0, x1, y in zip(train_data[0][:3], train_data[1][:3], train_data[2][:3]):
    print('Premise:', x0)
    print('Hypothesis:', x1)
    print('Label:', y)
    print()

Premise: A person on a horse jumps over a broken down airplane .
Hypothesis: A person is training his horse for a competition .
Label: 2

Premise: A person on a horse jumps over a broken down airplane .
Hypothesis: A person is at a diner , ordering an omelette .
Label: 1

Premise: A person on a horse jumps over a broken down airplane .
Hypothesis: A person is outdoors , on a horse .
Label: 0



The training set has about 550000 pairs, and the testing set has about 10000 pairs. The following shows that the three labels “entailment”, “contradiction”, and “neutral” are balanced in both the training set and the testing set.



In [8]:
test_data = read_snli(data_dir, is_train=False)
for data in [train_data, test_data]:
    print([data[2].count(i) for i in range(3)])

[183416, 183187, 182764]
[3368, 3237, 3219]


### 2. Defining a class for loading the dataset

Below we define a class for loading the SNLI dataset by inheriting from the `Dataset` class. The argument `num_steps` in the class constructor specifies the length of a text sequence so that each minibatch of sequences will have the same shape. In other words, tokens after the first num_steps ones in longer sequence are trimmed, while special tokens `<pad>` will be appended to shorter sequences until their length becomes num_steps. By implementing the `__getitem__` function, we can arbitrarily access the premise, hypothesis, and label with the index idx.



In [12]:
class SNLIDataset(torch.utils.data.Dataset):
    """A customized dataset class for SNLI dataset"""
    def __init__(self, dataset, num_steps, vocab=None):
        self.num_steps = num_steps
        premise_tokens = spy.tokenize(dataset[0])
        hypothesis_tokens = spy.tokenize(dataset[1])
        if vocab is None:
            self.vocab = spy.Vocab(
                premise_tokens + hypothesis_tokens, min_freq=5, reserved_tokens=['<pad>'])
        else: 
            self.vocab = vocab
        self.premises = self._pad(premise_tokens)
        self.hypotheses = self._pad(hypothesis_tokens)
        self.labels = torch.tensor(dataset[2])


    def _pad(self, lines):
        return torch.tensor([spy.truncate_pad(
                self.vocab[line], 
                num_steps=self.num_steps, 
                padding_token=self.vocab['<pad>']) for line in lines])
    

    def __getitem__(self, index):
        return (self.premises[index], self.hypotheses[index]), self.labels[index]
    

    def __len__(self):
        return len(self.labels)

### 3. Putting it all together

Now we can invoke the `read_snli` function and the `SNLIDataset` class to download the SNLI dataset and return DataLoader instances for both training and testing sets, together with the vocabulary of the training set. It is noteworthy that we must use the vocabulary constructed from the training set as that of the testing set. As a result, any new token from the testing set will be unknown to the model trained on the training set.



In [14]:
def load_data_snli(batch_size, num_steps=50):
    """Download the SNLI dataset and return data iterators and vocabulary"""
    data_dir = spy.download_extract(
        'https://nlp.stanford.edu/projects/snli/snli_1.0.zip', 
        '../../data', sha1_hash='9fcde07509c7e87ec61c640c1b2753d9041758e4')
    train_data = read_snli(data_dir, True)
    test_data = read_snli(data_dir, False)
    train_set = SNLIDataset(train_data, num_steps)
    test_set = SNLIDataset(test_data, num_steps)
    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)
    
    return train_iter, test_iter, train_set.vocab

Here we set the batch size to 128 and sequence length to 50, and invoke the `load_data_snli` function to get the data iterators and vocabulary. Then we print the vocabulary size.



In [15]:
train_iter, test_iter, vocab = load_data_snli(128, 50)
len(vocab)

18678

Now we print the shape of the first minibatch. Contrary to sentiment analysis, we have two inputs X[0] and X[1] representing pairs of premises and hypotheses.

In [16]:
for X, Y in train_iter:
    print(X[0].shape)
    print(X[1].shape)
    print(Y.shape)
    break

torch.Size([128, 50])
torch.Size([128, 50])
torch.Size([128])
